# 02 — Regras AML, ranking e validação da T1/T2

Objetivo: demonstrar como regras explicáveis podem ser aplicadas para priorizar transações/clientes e sustentar SAR.

Este notebook não tenta substituir o motor completo em `src/rules.py` e `src/alerts.py`. Ele funciona como caderno técnico de validação e leitura dos principais outputs.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
TZ = "America/Sao_Paulo"
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd()
DATA_PATH = ROOT / "data" / "raw" / "AML/FT Transaction Monitoring Case Study INC (2).xlsx"
T1_DIR = ROOT / "outputs" / "t1_suspects"
T2_DIR = ROOT / "outputs" / "t2_alert_system"
OUT_DIR = ROOT / "outputs" / "final_review"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base existe: {DATA_PATH.exists()}")
print(f"T1 existe: {T1_DIR.exists()} | T2 existe: {T2_DIR.exists()}")

## 1. Leitura dos dados e enriquecimentos

As regras combinam sinais transacionais com KYC e merchant. Isso permite avaliar risco de forma contextual, não apenas por valor.

In [ ]:
xl = pd.ExcelFile(DATA_PATH)
tx = xl.parse("Transactions")
kyc = xl.parse("KYC_Profiles")
merchants = xl.parse("Merchants")

tx["timestamp"] = pd.to_datetime(tx["timestamp"], errors="coerce")
tx["month"] = tx["timestamp"].dt.to_period("M").astype(str)

df = tx.merge(kyc, on="customer_id", how="left", suffixes=("", "_kyc"))
df = df.merge(merchants, on="merchant_id", how="left", suffixes=("", "_merchant"))

print(df.shape)
df.head(3)

## 2. Regras transacionais demonstrativas

Essas regras reproduzem a lógica central usada no case: sanções, país de risco, PEP, MCC de risco, e-commerce sem 3DS, device/IP anomaly, alto valor e self-merchant.

In [ ]:
rule_cols = []

def yes(col):
    return df[col].astype(str).str.lower().eq("yes") if col in df.columns else pd.Series(False, index=df.index)

df["r_sanctions_tx"] = yes("sanctions_screening_hit")
df["r_kyc_sanctions"] = yes("sanctions_list_hit")
df["r_pep"] = yes("pep")
df["r_ip_anomaly"] = yes("ip_anomaly")
df["r_device_rooted"] = yes("device_rooted")
df["r_cross_border"] = yes("cross_border")
df["r_high_risk_receiver_country"] = df.get("country_risk_receiver", "").astype(str).str.lower().eq("high")
df["r_mcc_high_risk"] = df.get("mcc_risk", "").astype(str).str.lower().eq("high")
df["r_merchant_high_risk"] = yes("merchant_high_risk_flag")
df["r_high_value_50k"] = df["amount_brl"].ge(50000)

df["r_ecommerce_no_3ds"] = (
    df["transaction_type"].eq("Card")
    & df.get("card_present", "").astype(str).eq("No")
    & df.get("auth_3ds", "").astype(str).isin(["No", "n/a", "nan", "None"])
)

if "owner_customer_id" in df.columns:
    df["r_self_merchant"] = df["owner_customer_id"].astype(str).eq(df["customer_id"].astype(str))
else:
    df["r_self_merchant"] = False

rule_cols = [c for c in df.columns if c.startswith("r_")]
df["transaction_rule_count"] = df[rule_cols].sum(axis=1)

coverage = df[rule_cols].sum().sort_values(ascending=False).reset_index()
coverage.columns = ["rule", "trigger_count"]
coverage.to_csv(OUT_DIR / "notebook_02_transaction_rule_coverage.csv", index=False)
coverage

## 3. Ranking transacional demonstrativo

A fila operacional prioriza concentração de sinais. Um alerta isolado pode ser triagem; múltiplos alertas independentes aumentam a prioridade.

In [ ]:
cols = [
    "transaction_id", "customer_id", "timestamp", "transaction_type", "amount_brl",
    "transaction_rule_count", "risk_rating", "pep", "country_risk_receiver",
    "mcc_risk", "sanctions_screening_hit", "cross_border", "ip_anomaly", "device_rooted"
]
cols = [c for c in cols if c in df.columns]

top_tx_demo = df.sort_values(["transaction_rule_count", "amount_brl"], ascending=[False, False])[cols].head(30)
top_tx_demo.to_csv(OUT_DIR / "notebook_02_demo_top30_transactions.csv", index=False)
top_tx_demo.head(10)

## 4. Regras cliente-mês demonstrativas

Algumas tipologias só aparecem no agregado: volume mensal fora do perfil, muitos cash-ins/cash-outs, velocity e valores redondos.

In [ ]:
df["is_round_amount"] = (df["amount_brl"].round(0) == df["amount_brl"]) | (df["amount_brl"] % 1000 < 1)
df["is_pix_cashin"] = df.get("pix_flow", "").astype(str).str.lower().eq("cash_in")
df["is_pix_cashout"] = df.get("pix_flow", "").astype(str).str.lower().eq("cash_out")

agg = df.groupby(["customer_id", "month"], dropna=False).agg(
    tx_count=("transaction_id", "count"),
    total_amount_brl=("amount_brl", "sum"),
    avg_amount_brl=("amount_brl", "mean"),
    max_amount_brl=("amount_brl", "max"),
    cashin_count=("is_pix_cashin", "sum"),
    cashout_count=("is_pix_cashout", "sum"),
    round_amount_count=("is_round_amount", "sum"),
    tx_rule_count_sum=("transaction_rule_count", "sum"),
).reset_index()

kyc_keep = ["customer_id", "annual_income_brl", "risk_rating", "pep", "declared_occupation"]
kyc_keep = [c for c in kyc_keep if c in kyc.columns]
agg = agg.merge(kyc[kyc_keep], on="customer_id", how="left")
agg["monthly_income_est"] = agg["annual_income_brl"] / 12
agg["amount_income_ratio"] = agg["total_amount_brl"] / agg["monthly_income_est"].replace({0: np.nan})

# Regras mensais demonstrativas
agg["r_month_out_of_profile"] = agg["amount_income_ratio"].ge(2).fillna(False)
agg["r_month_high_velocity"] = agg["tx_count"].ge(60)
agg["r_month_many_cashouts"] = agg["cashout_count"].ge(30)
agg["r_month_many_cashins"] = agg["cashin_count"].ge(9)
agg["r_month_round_amounts"] = agg["round_amount_count"].ge(10)
agg["r_month_pass_through"] = agg["r_month_high_velocity"] & agg["r_month_many_cashins"] & agg["r_month_many_cashouts"]

month_rule_cols = [c for c in agg.columns if c.startswith("r_month_")]
agg["month_rule_count"] = agg[month_rule_cols].sum(axis=1)
agg["weak_label_demo"] = (agg["month_rule_count"] >= 3).astype(int)

agg.sort_values(["month_rule_count", "total_amount_brl"], ascending=[False, False]).head(10)

## 5. Comparação com os outputs finais da T1/T2

Aqui eu valido que os arquivos entregues existem e faço uma leitura rápida dos rankings finais usados no relatório.

In [ ]:
required_outputs = [
    T1_DIR / "02_suspicious_transactions_top30.csv",
    T1_DIR / "03_suspicious_clients_top30.csv",
    T1_DIR / "07_SAR_draft_C101208.md",
    T2_DIR / "01_alert_rules_catalog_t2.csv",
]

validation = pd.DataFrame({
    "file": [str(p) for p in required_outputs],
    "exists": [p.exists() for p in required_outputs],
    "size_bytes": [p.stat().st_size if p.exists() else 0 for p in required_outputs],
})
validation.to_csv(OUT_DIR / "notebook_02_required_outputs_validation.csv", index=False)
validation

In [ ]:
final_top_tx = pd.read_csv(T1_DIR / "02_suspicious_transactions_top30.csv")
final_top_clients = pd.read_csv(T1_DIR / "03_suspicious_clients_top30.csv")
rule_catalog = pd.read_csv(T2_DIR / "01_alert_rules_catalog_t2.csv")

print("Top transações finais:", final_top_tx.shape)
print("Top clientes finais:", final_top_clients.shape)
print("Regras catalogadas:", rule_catalog.shape)

final_top_clients.head(10)

In [ ]:
plt.figure(figsize=(8, 4.5))
coverage.head(12).sort_values("trigger_count").plot(kind="barh", x="rule", y="trigger_count", legend=False, ax=plt.gca())
plt.title("Cobertura das regras transacionais demonstrativas")
plt.xlabel("Quantidade de acionamentos")
plt.ylabel("Regra")
plt.tight_layout()
plt.savefig(OUT_DIR / "notebook_02_rule_coverage_demo.png", dpi=160)
plt.show()

## 6. Conclusão

O motor de regras permite justificar os alertas de forma auditável. O ranking final do case usa essa lógica para priorizar clientes e transações, e o SAR escolhido é sustentado por combinação de sinais, timeline e materialidade.